In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
%sql
DROP TABLE IF EXISTS promotion.DIM_EmailNotification;
CREATE TABLE IF NOT EXISTS promotion.DIM_EmailNotification
(
  EmailNotificationID INT, 
  SenderEmail STRING,
  RecipientEmails STRING,
  CCEmails STRING,
  EmailSubject STRING,
  BasicQuery STRING,
  HTMLContent STRING,
  SortByColumn STRING,
  SortByAscending STRING,
  CreatedDate Date,
  CreatedBy String,
  UpdatedDate Date,
  UpdatedBy String,
  Active boolean
)
USING delta
LOCATION 'dbfs:/mnt/silver/DIMEmailNotification';

TRUNCATE TABLE promotion.DIM_EmailNotification;

In [0]:
basic_query = '''with CSID as 
( 
Select Distinct CoolSculptingID ,count(Distinct ShipToAccountUUID) AS ShipToCount,count(distinct IAD.PatientClassificationUUID) As PatientClassCount,
        count(Distinct IAD.SoldToAccountID) AS SoldToCount
From promotion.fact_invoiceaddendumdetails AS IAD
inner join promotion.dim_patientclassification as pc         
        on pc.PatientClassificationUUID = IAD.PatientClassificationUUID        
Where CoolSculptingID <> "MISSING" AND ifnull(ExceptionMoxieCaseNumber,"") = ""
AND VersionID = 1
AND pc.PatientClassificationName = "PerPatientFee"
group by all Having count(Distinct ShipToAccountUUID) > 1 )

Select CoolSculptingID,ScannedId,ShipToAccountId,SoldToAccountID,PatientClassificationName,CycleDate,CycleCount,UpdatedBy,SubscriptionStartDate,SubscriptionEndDate,
       SoldToComment
FROM
(
SELECT  CASE WHEN SoldToCount > 1 THEN "DiffSoldTo" ELSE "SameSoldTo" END AS SoldToComment,
        Ia.CoolSculptingID,dc.ScannedId,da.ShipToAccountId,Ia.SoldToAccountID,pc.PatientClassificationName,Ia.CycleDate,Ia.CycleCount,CSB.UpdatedBy,
        CSB.SubscriptionStartDate,CSB.SubscriptionEndDate
From promotion.fact_invoiceaddendumdetails As Ia 
Left join promotion.dim_consumer AS dc        
        ON dc.CoolsculptingId = IA.CoolSculptingID 
left join promotion.fact_consumersubscription AS CSB
        ON Ia.ConsumerSubscriptionUUID = CSB.ConsumerSubscriptionUUID
        AND CSB.VersionID = 1        
inner join promotion.dim_account as da         
        on Ia.ShipToAccountUUID = da.ShipToAccountUUID 
inner join promotion.dim_patientclassification as pc         
        on pc.PatientClassificationUUID = Ia.PatientClassificationUUID        
inner join CSID AS CS         
        on CS.CoolSculptingID = Ia.CoolSculptingID 
Where Ia.VersionID = 1 

)

ORDER BY CycleDate DESC,SoldToComment DESC,ScannedId ASC,SoldToAccountID,SubscriptionStartDate ASC,PatientClassificationName
'''
subject='PPP Validation notifications - User having multiple Subscription Report'
sender_email = 'IM_EmailNotificationagn-dl-coolconnectapp-support@abbvie.com'
recipient_emails = 'brandon.lemesh@allergan.com,piya.guerrero@allergan.com,alexis.barta@allergan.com,jade.nightingale@allergan.com'
cc_emails = 'neeta.patil@abbvie.com,subina.sudhakaran@abbvie.com,srinadhareddy.yarram@abbvie.com'
html_Content = '''
<html> 
  <head> Hi Team,  <br>
  </head>
  <body>
        <p> 
           <b><u>User having multiple Subscription with Different ShipTo:</b></u><br>
           <basic_query>
           Thanks,<br>
            AGN-DL-CoolConnectApp-Support 
        </p>
  </body>
</html>
'''
sortByColumn = 'CycleDate,CoolSculptingID,PatientClassificationName'
sortByAscending = 'False,True,True'

s_sql = '''INSERT INTO promotion.DIM_EmailNotification 
            (EmailNotificationID,SenderEmail,RecipientEmails,CCEmails,EmailSubject,BasicQuery,HTMLContent,SortByColumn,SortByAscending,
            CreatedDate,CreatedBy,UpdatedDate,UpdatedBy,Active)
            values
            (1,'{}', '{}', '{}','{}', '{}','{}', '{}','{}',
            Current_Date(),'AAIOT2277-EmailSetUp',Current_Date(),'AAIOT2277-EmailSetUp',True)'''.format(sender_email,recipient_emails,cc_emails,subject,basic_query,html_Content,sortByColumn,sortByAscending)
print(s_sql)
spark.sql(s_sql)

In [0]:
%sql
select * From promotion.DIM_EmailNotification